# TAREA 6
María Fernanda Gómez Flores  
Carlos Hassan Hernández Rohan  
Portafolios de Inversión

## PUNTO A

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

### 1. MÍNIMA SEMIVARIANZA

In [2]:
# Descarga de Datos
prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
                     start='2016-01-01', end='2026-10-15')['Close']

prices.isna().sum()

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_12942/1304681109.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
[*********************100%***********************]  6 of 6 completed


Ticker
CLX     0
COST    0
KHC     0
KO      0
PG      0
WMT     0
dtype: int64

In [3]:
# Calculo de rendimientos
rets=prices.pct_change().dropna()

In [4]:
def opt_min_semivar(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular matriz de correlación
    corr = rets.corr()

    #Calcular downside risk
    below_zero = (rets[rets<0]).fillna(0)
    downside = np.array(below_zero.std())

    #Calcular matriz de semivarianza
    n_assets = len(prices.columns)
    semivar_matrix = (downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr

    # Funcion.
    semivar = lambda w: w.T @ semivar_matrix @ w

    #Variables iniciales (pesos)
    w_0 = np.ones(n_assets) / n_assets

    # Restricciones.
    restr = lambda w: sum(w) - 1

    #rangos de las variables de decisión
    rangos = [(0,1)] * n_assets

    # tolerancia
    tol = 1e-32

    #Optimizar
    results = minimize(fun=semivar, x0=w_0, bounds=rangos,
                       constraints={'fun': restr, 'type': 'eq'}, tol=tol)

    return results.x

In [5]:
# Verificar resultados de la función
w_min_semivar=opt_min_semivar(prices)
dict(zip(prices.columns,w_min_semivar))

{'CLX': np.float64(0.08719370387114594),
 'COST': np.float64(0.1508835446412648),
 'KHC': np.float64(0.019272940190893997),
 'KO': np.float64(0.3342833635442749),
 'PG': np.float64(0.21097903846990135),
 'WMT': np.float64(0.19738740928251908)}

In [6]:
below_zero=(rets[rets<0]).fillna(0)
below_zero

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-01-06,-0.000472,-0.009254,0.000000,-0.005405,-0.009666,0.000000
2016-01-07,-0.010942,-0.022910,-0.022168,-0.016540,-0.008734,0.000000
2016-01-08,0.000000,-0.017504,-0.010014,-0.002643,-0.015677,-0.022913
2016-01-11,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
2026-10-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2026-10-09,0.000000,-0.001055,-0.009342,0.000000,0.000000,0.000000
2026-10-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [7]:
downside=np.array(below_zero.std())
downside

array([0.00998865, 0.0084987 , 0.01128803, 0.00725476, 0.0073185 ,
       0.00839043])

In [8]:
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix=(downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*rets.corr()
semivar_matrix

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000100,0.000027,0.000038,0.000027,0.000038,0.000025
COST,0.000027,0.000072,0.000026,0.000024,0.000026,0.000041
KHC,0.000038,0.000026,0.000127,0.000038,0.000037,0.000026
KO,0.000027,0.000024,0.000038,0.000053,0.000033,0.000021
PG,0.000038,0.000026,0.000037,0.000033,0.000054,0.000026
WMT,0.000025,0.000041,0.000026,0.000021,0.000026,0.000070


In [9]:
semivar_port=lambda w: w.T @ semivar_matrix @ w

In [10]:
# Obtener semivarianza mínima
semivar_port(w_min_semivar)

np.float64(3.541243250582367e-05)

#### Comprobación Montecarlo

In [11]:
# Simular portafolios con montecarlo
np.random.seed(42)
n_sim=10000
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [12]:
# Obtener la mínima semivarianza
semivars_port=[semivar_port(w_i) for w_i in w_random]
np.min(semivars_port)

np.float64(3.563834300245848e-05)

In [13]:
# Obtener los pesos del portafolio con mínima semivarianza
w_semivar=w_random[np.argmin(semivars_port)]
dict(zip(prices.columns, w_semivar))

{'CLX': np.float64(0.04523198269375449),
 'COST': np.float64(0.18032970063786),
 'KHC': np.float64(0.02431282278193316),
 'KO': np.float64(0.34349231541965103),
 'PG': np.float64(0.19048463084264106),
 'WMT': np.float64(0.2161485476241604)}

In [14]:
# Comparar resultados
comparacion=pd.DataFrame({
    'SciPy':w_min_semivar*100,
    'Montecarlo':w_semivar*100
},index=prices.columns)

comparacion

,SciPy,Montecarlo
Ticker,,
CLX,8.719370,4.523198
COST,15.088354,18.032970
KHC,1.927294,2.431282
KO,33.428336,34.349232
PG,21.097904,19.048463
WMT,19.738741,21.614855


**Conclusión**  
Con SciPy obtuvimos una semivarianza de 0.0000354124 y con Montecarlo de 0.0000356383. Los resultados quedaron bastante cercanos, aunque SciPy encontró un valor un poco menor. Esto tiene sentido porque Montecarlo solamente busca entre los portafolios que se simularon. Estos valores corresponden a la corrida que hicimos y pueden cambiar si descargamos los precios otra vez.

### 2. MÁXIMO OMEGA

In [15]:
def opt_max_omega(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular downside y upside risk
    below_zero=rets[rets<0].fillna(0)
    above_zero=rets[rets>0].fillna(0)
    upside_risk=np.array(above_zero.std())
    downside_risk=np.array(below_zero.std())

    # Calcular Omega individual
    omega=upside_risk/downside_risk

    # Funcion
    func_omega=lambda w: -sum(w*omega)

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    n_assets=len(prices.columns)
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-32

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_omega,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    return results.x

In [16]:
# Obtener los pesos
w_max_omega=opt_max_omega(prices)
dict(zip(prices.columns,w_max_omega))

{'CLX': np.float64(8.537637339511252e-20),
 'COST': np.float64(5.359439907723842e-20),
 'KHC': np.float64(3.527769728877312e-24),
 'KO': np.float64(7.816385030662051e-24),
 'PG': np.float64(1.200046277817617e-22),
 'WMT': np.float64(1.0)}

#### Comprobación Montecarlo

In [17]:
below_zero=rets[rets<0].fillna(0)
above_zero=rets[rets>0].fillna(0)

In [18]:
upside_risk=np.array(above_zero.std())
upside_risk

array([0.00907326, 0.00843025, 0.01005744, 0.00695529, 0.00740139,
       0.00898112])

In [19]:
downside_risk=np.array(below_zero.std())
downside_risk

array([0.00998865, 0.0084987 , 0.01128803, 0.00725476, 0.0073185 ,
       0.00839043])

In [20]:
omega=upside_risk/downside_risk
omega

array([0.90835677, 0.99194584, 0.89098313, 0.95872145, 1.01132677,
       1.07040097])

In [21]:
# Obtener el maximo omega
func_omega=lambda w: sum(w*omega)

In [22]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [23]:
# Obtener el maximo omega
omegas_port=[func_omega(w_i) for w_i in w_random]
np.max(omegas_port)

np.float64(1.0574775642962502)

In [24]:
# Obtener los pesos del portafolio con maximo omega
w_omega_mc=w_random[np.argmax(omegas_port)]
dict(zip(rets.columns,w_omega_mc))

{'CLX': np.float64(0.03399832366440288),
 'COST': np.float64(0.012029943661879684),
 'KHC': np.float64(0.01443209502964946),
 'KO': np.float64(0.033455007736670934),
 'PG': np.float64(0.002450246946761601),
 'WMT': np.float64(0.9036343829606355)}

In [25]:
# Comparar resultados
comparacion_omega=pd.DataFrame({
    'SciPy':w_max_omega*100,
    'Montecarlo':w_omega_mc*100
},index=prices.columns)
comparacion_omega

,SciPy,Montecarlo
Ticker,,
CLX,8.537637e-18,3.399832
COST,5.359440e-18,1.202994
KHC,3.527770e-22,1.443210
KO,7.816385e-22,3.345501
PG,1.200046e-20,0.245025
WMT,1.000000e+02,90.363438


**Conclusión**  
*Aquí escribimos la comparación cuando tengamos los resultados de los dos métodos.*

### 3. SEMIVARIANZA OBJETIVO

In [26]:
# Descarga de Precios del Benchmark
benchmark = yf.download('KXI', start='2016-01-01', end='2026-10-15')['Close']
benchmark

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_12942/4160543330.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  benchmark = yf.download('KXI', start='2016-01-01', end='2026-10-15')['Close']
[*********************100%***********************]  1 of 1 completed


Ticker,KXI
Date,
2016-01-04,35.621937
2016-01-05,35.711323
2016-01-06,35.396503
2016-01-07,34.933971
2016-01-08,34.654137
...,...
2026-10-08,67.309998
2026-10-09,67.400002
2026-10-12,67.889999


In [27]:
# Obtener rendimientos del benchmark
rets_benchmark=benchmark.pct_change().dropna()
rets_benchmark

Ticker,KXI
Date,
2016-01-05,0.002509
2016-01-06,-0.008816
2016-01-07,-0.013067
2016-01-08,-0.008010
2016-01-11,0.008636
...,...
2026-10-08,0.021086
2026-10-09,0.001337
2026-10-12,0.007270


In [28]:
# Auditoría de fechas
print(prices.index.min(), prices.index.max())
print(benchmark.index.min(), benchmark.index.max())
print(prices.isna().sum())
print(benchmark.isna().sum())

2016-01-04 00:00:00 2026-10-14 00:00:00
2016-01-04 00:00:00 2026-10-14 00:00:00
Ticker
CLX     0
COST    0
KHC     0
KO      0
PG      0
WMT     0
dtype: int64
Ticker
KXI    0
dtype: int64


In [29]:
def opt_target_semivar(prices: pd.DataFrame, benchmark: pd.Series):
    #Calcular los rendimientos
    rets=prices.pct_change().dropna()
    rets_benchmark=benchmark.pct_change().dropna()

    # Sustraer rendimiento del benchmark de rendimientos individuales
    alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()

    # Separar los rendimientos por debajo de cero
    below_zero_t=(alpha[alpha<0]).fillna(0)

    # Calcular target downside risk
    downside_risk_t=np.array(below_zero_t.std())

    # Calcular la matriz de target semivariance
    corr=alpha.corr()
    n_assets=len(prices.columns)
    semivar_matrix=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*corr

    # Funcion
    func_target=lambda w: w.T @ semivar_matrix @ w

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-32

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_target,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    return results.x

In [30]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_target_semivar=opt_target_semivar(prices,benchmark)
dict(zip(prices.columns,w_target_semivar))

{'CLX': np.float64(0.04129651082536334),
 'COST': np.float64(0.15756780122007844),
 'KHC': np.float64(0.0772574563913047),
 'KO': np.float64(0.33541428294637055),
 'PG': np.float64(0.27536875905190467),
 'WMT': np.float64(0.11309518956497834)}

#### Comprobación Montecarlo

In [31]:
# Sustraer rendimiento del benchmark de rendimientos individuales
alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()
alpha

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.011939,-0.000065,0.006982,0.001028,0.000680,0.021246
2016-01-06,0.008343,-0.000438,0.010723,0.003410,-0.000850,0.018828
2016-01-07,0.002125,-0.009842,-0.009101,-0.003473,0.004333,0.036356
2016-01-08,0.012308,-0.009494,-0.002003,0.005368,-0.007667,-0.014903
2016-01-11,0.003252,0.008523,0.006677,-0.006950,0.000578,0.002066
...,...,...,...,...,...,...
2026-10-08,-0.006124,-0.015069,0.001662,0.001636,-0.002347,0.001103
2026-10-09,0.002977,-0.002392,-0.010679,0.001853,0.002913,0.005808
2026-10-12,-0.000707,0.005952,0.000813,0.012491,0.005757,0.008267


In [32]:
# Separar los rendimientos por debajo de cero
below_zero_t=(alpha[alpha<0]).fillna(0)
below_zero_t

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.000000,-0.000065,0.000000,0.000000,0.000000,0.000000
2016-01-06,0.000000,-0.000438,0.000000,0.000000,-0.000850,0.000000
2016-01-07,0.000000,-0.009842,-0.009101,-0.003473,0.000000,0.000000
2016-01-08,0.000000,-0.009494,-0.002003,0.000000,-0.007667,-0.014903
2016-01-11,0.000000,0.000000,0.000000,-0.006950,0.000000,0.000000
...,...,...,...,...,...,...
2026-10-08,-0.006124,-0.015069,0.000000,0.000000,-0.002347,0.000000
2026-10-09,0.000000,-0.002392,-0.010679,0.000000,0.000000,0.000000
2026-10-12,-0.000707,0.000000,0.000000,0.000000,0.000000,0.000000


In [33]:
# Calcular target downside risk
downside_risk_t=np.array(below_zero_t.std())
downside_risk_t

array([0.00865549, 0.00684588, 0.00956583, 0.00473383, 0.00478274,
       0.00712827])

In [34]:
# Calcular la matriz de target semivariance
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix_t=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*alpha.corr()
semivar_matrix_t

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,7.491753e-05,0.000003,0.000007,7.313629e-07,0.000011,0.000004
COST,2.607565e-06,0.000047,-0.000006,-2.740177e-06,-0.000001,0.000018
KHC,7.057945e-06,-0.000006,0.000092,3.781595e-06,0.000002,-0.000003
KO,7.313629e-07,-0.000003,0.000004,2.240915e-05,0.000003,-0.000003
PG,1.118013e-05,-0.000001,0.000002,2.808971e-06,0.000023,0.000001
WMT,4.002827e-06,0.000018,-0.000003,-3.128310e-06,0.000001,0.000051


In [35]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [36]:
# Obtener la mínima target semivariance
func_target = lambda w: w.T @ semivar_matrix_t @ w
mts=[func_target(w_i)for w_i in w_random]
np.min(mts)

np.float64(7.876672326150037e-06)

In [37]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_tarjet_semivar = w_random[np.argmin(mts)]
dict(zip(rets.columns, w_tarjet_semivar))

{'CLX': np.float64(0.05018925389310219),
 'COST': np.float64(0.13631386488141561),
 'KHC': np.float64(0.07003453981745411),
 'KO': np.float64(0.3520023944212645),
 'PG': np.float64(0.25385946443983465),
 'WMT': np.float64(0.13760048254692883)}

In [ ]:
# Comparar resultados
comparacion_target=pd.DataFrame({
    'SciPy':w_target_semivar*100,
    'Montecarlo':w_tarjet_semivar*100
},index=prices.columns)

comparacion_target

,SciPy,Montecarlo
Ticker,,
CLX,4.129651,5.018925
COST,15.756780,13.631386
KHC,7.725746,7.003454
KO,33.541428,35.200239
PG,27.536876,25.385946
WMT,11.309519,13.760048


**Conclusión**  
*Aquí escribimos qué diferencias encontramos con el benchmark cuando tengamos los resultados.*